
# TMaze: binned linearization using preprocessed positions and metadata ports

This notebook does the following for a single TMaze session:

1. Load:
   * `*_TMaze_preprocessed_positions.csv`
   * `*_TMaze_trials.csv`
   * `TMaze_metadata.csv` (for `port_rois`) and `dlc_trim_info.csv` (for miniscope `start_frame`).
2. Use `x_smooth`, `y_smooth` from the preprocessed positions as the 2D trajectory.
3. Adjust trial timing to miniscope-trimmed frame coordinates (frame 0 = `start_frame` from `dlc_trim_info.csv`).
4. Build 1D *linearized* position along the base–left, base–right, and left–right paths, using **port coordinates from the metadata**.
5. Discretize each path into spatial **bins** (e.g. 30 bins per path) and convert each trial into a sequence of bin IDs instead of raw coordinates.
6. Make diagnostic plots to verify per–trial geometry and binning.
7. Optionally save a CSV with linearized and binned trial coordinates in the same directory as the session CSVs.


In [ ]:

import numpy as np
import pandas as pd
import os
import ast

import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 100


In [ ]:

def _project_onto_segment(p, a, b):
    """Project point p onto segment a-b."""
    ab = b - a
    ab2 = np.dot(ab, ab)
    if ab2 == 0:
        return a, 0.0, np.sum((p - a)**2)
    t = np.dot(p - a, ab) / ab2
    t_clipped = np.clip(t, 0.0, 1.0)
    proj = a + t_clipped * ab
    dist2 = np.sum((p - proj)**2)
    return proj, t_clipped, dist2


def linearize_positions_along_polyline(location, poly_points):
    """
    location    : (N, 2) array of [x, y] positions
    poly_points : (M, 2) array of [x, y] points defining the path.

    Returns
    -------
    linear_pos  : (N,) array, arc-length coordinate along the polyline.
    """
    loc = np.asarray(location, float)
    poly = np.asarray(poly_points, float)

    seg_vecs = poly[1:] - poly[:-1]
    seg_len = np.linalg.norm(seg_vecs, axis=1)
    cum_len = np.concatenate([[0.0], np.cumsum(seg_len)])

    N = loc.shape[0]
    linear_pos = np.full(N, np.nan)

    for i in range(N):
        p = loc[i]
        if np.any(np.isnan(p)):
            continue

        best_dist2 = np.inf
        best_s = np.nan

        for s_idx in range(len(seg_len)):
            a = poly[s_idx]
            b = poly[s_idx + 1]
            _, t, d2 = _project_onto_segment(p, a, b)
            if d2 < best_dist2:
                best_dist2 = d2
                best_s = cum_len[s_idx] + t * seg_len[s_idx]

        linear_pos[i] = best_s

    return linear_pos


In [ ]:

def load_trim_info(trim_csv_path):
    """Load dlc_trim_info.csv and return a Series with at least 'start_frame'."""
    trim_df = pd.read_csv(trim_csv_path)
    if trim_df.empty:
        raise ValueError("dlc_trim_info.csv is empty.")
    row = trim_df.iloc[0]
    if "start_frame" not in trim_df.columns:
        raise ValueError("dlc_trim_info.csv must contain a 'start_frame' column.")
    return row


def load_trials(trials_csv_path):
    """Load TMaze_trials.csv; expects start_frame, end_frame, trial_type."""
    trials = pd.read_csv(trials_csv_path)
    required = {"start_frame", "end_frame", "trial_type"}
    missing = required - set(trials.columns)
    if missing:
        raise ValueError(f"Trials CSV is missing columns: {missing}")
    return trials


def load_preprocessed_positions(preproc_csv_path,
                                x_col="x_smooth",
                                y_col="y_smooth"):
    """
    Load TMaze_preprocessed_positions.csv and return coords + frames.

    Returns
    -------
    coords : (N, 2) array from [x_col, y_col]
    frames : (N,) array of int frame indices (original DLC space)
    df     : full DataFrame (for debugging)
    """
    df = pd.read_csv(preproc_csv_path)
    for c in ["frame", x_col, y_col]:
        if c not in df.columns:
            raise ValueError(f"Preprocessed positions CSV missing column: {c}")
    coords = df[[x_col, y_col]].to_numpy(dtype=float)
    frames = df["frame"].to_numpy(dtype=int)
    return coords, frames, df


def load_ports_from_metadata(metadata_csv_path):
    """
    Read TMaze_metadata.csv and extract port centers from 'port_rois'.

    `port_rois` is expected to be a stringified dict, e.g.:
        {'base': (x_b, y_b, r_b), 'left': (x_l, y_l, r_l), 'right': (x_r, y_r, r_r)}

    We take the (x, y) as port coordinates.
    Returns a dict: {'base': np.array([x_b, y_b]), ...}
    """
    meta = pd.read_csv(metadata_csv_path)
    row = meta.loc[meta["Unnamed: 0"] == "port_rois"]
    if row.empty:
        raise ValueError("metadata CSV has no 'port_rois' row.")
    port_str = row["value"].iloc[0]
    ports_raw = ast.literal_eval(port_str)

    ports = {}
    for key, val in ports_raw.items():
        # val is e.g. (x, y, radius)
        x, y = float(val[0]), float(val[1])
        ports[key] = np.array([x, y], dtype=float)
    return ports


def make_polylines_from_ports(ports):
    """
    Given ports dict {'base': [x,y], 'left': [x,y], 'right': [x,y]},
    build polylines for BL, BR, LR paths.
    """
    for k in ["base", "left", "right"]:
        if k not in ports:
            raise ValueError(f"ports dict is missing key: {k!r}")
    base = ports["base"]
    left = ports["left"]
    right = ports["right"]

    poly_BL = np.vstack([base, left])
    poly_BR = np.vstack([base, right])
    poly_LR = np.vstack([left, right])

    return base, left, right, poly_BL, poly_BR, poly_LR


In [ ]:

def adjust_trials_for_trim(trials, trim_start_frame, n_frames_trimmed=None):
    """
    Re-express trial start/end frames relative to trimmed miniscope start.

    Adds columns:
        - trim_start
        - trim_end
    """
    trials_adj = trials.copy()
    trim_start_frame = int(trim_start_frame)

    trials_adj["trim_start"] = trials_adj["start_frame"] - trim_start_frame
    trials_adj["trim_end"]   = trials_adj["end_frame"]   - trim_start_frame

    if n_frames_trimmed is not None:
        trials_adj["trim_start"] = trials_adj["trim_start"].clip(0, n_frames_trimmed - 1)
        trials_adj["trim_end"]   = trials_adj["trim_end"].clip(0, n_frames_trimmed - 1)

    return trials_adj


def build_trimmed_location_from_preprocessed(coords_full,
                                             frames_full,
                                             trim_start_frame):
    """
    Slice preprocessed coordinates to trimmed coordinates
    (frame 0 == trim_start_frame).
    """
    trim_start_frame = int(trim_start_frame)

    if trim_start_frame < frames_full.min() or trim_start_frame > frames_full.max():
        raise ValueError("trim_start_frame is outside range of preprocessed frames.")

    mask = frames_full >= trim_start_frame
    loc_trim = coords_full[mask]

    return loc_trim


In [ ]:

# Trial-type groups, matched to TMaze_trials.csv
bl_types = {"base_to_left", "left_to_base"}
br_types = {"base_to_right", "right_to_base"}
lr_types = {"left_to_right", "right_to_left"}


def get_path_for_trial_type(trial_type):
    """Return the string key for which path a trial_type uses."""
    if trial_type in bl_types:
        return "BL"
    elif trial_type in br_types:
        return "BR"
    elif trial_type in lr_types:
        return "LR"
    else:
        raise ValueError(f"Unknown trial_type: {trial_type}")


def linearize_and_bin_trials(location_trim,
                             trials_adj,
                             base,
                             left,
                             right,
                             poly_BL,
                             poly_BR,
                             poly_LR,
                             n_bins=30,
                             save=False,
                             out_dir=None,
                             out_basename="TMaze_linearized_binned_trials",
                             max_trials_per_type_for_plots=3):
    """
    Linearize TMaze coordinates for each trial, discretize into spatial bins,
    and optionally save results.

    Parameters
    ----------
    location_trim : (N_trim, 2)
        2D trajectory in trimmed frame coordinates (index 0 == trim start).
    trials_adj : DataFrame
        Must contain trim_start, trim_end, trial_type.
    base, left, right : np.array
        Port centers for plotting.
    poly_BL, poly_BR, poly_LR : np.ndarray
        Polylines for each path.
    n_bins : int
        Number of bins per path. Global bin IDs = path_offset + local_bin.
    save : bool
        If True, writes a CSV to out_dir.
    out_dir : str or None
        Directory to write the CSV. If None and save=True, uses current dir.
    out_basename : str
        Base name of output CSV.
    max_trials_per_type_for_plots : int
        Number of example trials per type for diagnostic plots.

    Returns
    -------
    lin_df : DataFrame
        Columns:
            trial_index, trial_type,
            frame_in_trial, frame_global_trim,
            path, bin_local, bin_global, linpos
    bin_edges : dict
        {'BL': edges_BL, 'BR': edges_BR, 'LR': edges_LR}
    """
    loc = np.asarray(location_trim, float)
    if loc.ndim != 2 or loc.shape[1] != 2:
        raise ValueError("location_trim must be (N_trim, 2)")

    # Full-series linearized positions
    linpos_BL = linearize_positions_along_polyline(loc, poly_BL)
    linpos_BR = linearize_positions_along_polyline(loc, poly_BR)
    linpos_LR = linearize_positions_along_polyline(loc, poly_LR)

    # Define bin edges per path (0 .. max distance)
    bin_edges = {}
    bin_indices_full = {}

    # Helper to build edges and full-path bin indices
    def make_bins(linpos):
        max_val = np.nanmax(linpos)
        edges = np.linspace(0.0, max_val, n_bins + 1)
        # digitize returns 1..n_bins inclusive; subtract 1 to get 0..n_bins-1
        b = np.digitize(linpos, edges, right=False) - 1
        b = np.clip(b, 0, n_bins - 1)
        return edges, b

    edges_BL, bins_BL = make_bins(linpos_BL)
    edges_BR, bins_BR = make_bins(linpos_BR)
    edges_LR, bins_LR = make_bins(linpos_LR)

    bin_edges["BL"] = edges_BL
    bin_edges["BR"] = edges_BR
    bin_edges["LR"] = edges_LR

    bin_indices_full["BL"] = bins_BL
    bin_indices_full["BR"] = bins_BR
    bin_indices_full["LR"] = bins_LR

    # Global bin offsets so every bin has a unique ID across paths
    offset_BL = 0
    offset_BR = n_bins
    offset_LR = 2 * n_bins
    path_offset = {"BL": offset_BL, "BR": offset_BR, "LR": offset_LR}

    rows = []
    for idx, tr in trials_adj.iterrows():
        ttype = tr["trial_type"]
        path_key = get_path_for_trial_type(ttype)

        start = int(tr["trim_start"])
        end   = int(tr["trim_end"])

        if start < 0 or end >= loc.shape[0] or end <= start:
            continue

        if path_key == "BL":
            lin_full = linpos_BL
            bins_full = bins_BL
        elif path_key == "BR":
            lin_full = linpos_BR
            bins_full = bins_BR
        else:  # "LR"
            lin_full = linpos_LR
            bins_full = bins_LR

        seg_lin = lin_full[start:end+1]
        seg_bin_local = bins_full[start:end+1]
        seg_bin_global = path_offset[path_key] + seg_bin_local

        for k in range(len(seg_lin)):
            rows.append({
                "trial_index": idx,
                "trial_type": ttype,
                "frame_in_trial": k,
                "frame_global_trim": start + k,
                "path": path_key,
                "bin_local": int(seg_bin_local[k]),
                "bin_global": int(seg_bin_global[k]),
                "linpos": float(seg_lin[k]),
            })

    lin_df = pd.DataFrame(rows)

    # --- Diagnostic plots ---
    unique_types = trials_adj["trial_type"].unique()
    for ttype in unique_types:
        tr_idxs = trials_adj.index[trials_adj["trial_type"] == ttype].tolist()
        if not tr_idxs:
            continue
        tr_idxs = tr_idxs[:max_trials_per_type_for_plots]

        for tid in tr_idxs:
            tr = trials_adj.loc[tid]
            start = int(tr["trim_start"])
            end   = int(tr["trim_end"])
            if start < 0 or end >= loc.shape[0] or end <= start:
                continue

            coords = loc[start:end+1]
            seg_lin = lin_df[lin_df["trial_index"] == tid]["linpos"].to_numpy()
            seg_bins = lin_df[lin_df["trial_index"] == tid]["bin_local"].to_numpy()

            fig, axes = plt.subplots(1, 2, figsize=(11, 4))

            # 2D path with ports
            axes[0].plot(loc[:, 0], loc[:, 1], alpha=0.2, label="all (trim)")
            axes[0].plot(coords[:, 0], coords[:, 1], "-o", label=f"trial {tid}")
            axes[0].scatter(
                [base[0], left[0], right[0]],
                [base[1], left[1], right[1]],
                marker="x", s=60, label="ports"
            )
            axes[0].set_title(f"{ttype} – trial {tid} path (trim coords)")
            axes[0].set_xlabel("x")
            axes[0].set_ylabel("y")
            axes[0].legend()

            # Continuous linpos + bins
            axes[1].plot(seg_lin, "-o", label="linpos")
            axes[1].step(
                np.arange(len(seg_bins)),
                seg_bins,
                where="mid",
                label="bin_local",
                alpha=0.7
            )
            axes[1].set_title(f"{ttype} – trial {tid} linearized & binned")
            axes[1].set_xlabel("frame in trial")
            axes[1].set_ylabel("value")
            axes[1].grid(True)
            axes[1].legend()

            plt.tight_layout()
            plt.show()

    # --- Optional save ---
    if save:
        if out_dir is None:
            out_dir = os.getcwd()
        os.makedirs(out_dir, exist_ok=True)
        out_csv = os.path.join(out_dir, out_basename + ".csv")
        lin_df.to_csv(out_csv, index=False)
        print("Saved linearized + binned trial coordinates to:", out_csv)

    return lin_df, bin_edges


In [ ]:

# === Example usage for one session ===
# Edit these paths for your session, then uncomment & run.

base_dir         = r"/media/toor/T7Shield/Mouse1637/Round1.5/TMaze/2024-06-10T17_53_09/"  # directory that contains the CSVs
trim_info_path   = os.path.join(base_dir, "dlc_trim_info.csv")
trials_csv_path  = os.path.join(base_dir,"dlc", "2024-06-10_17-53-09_TMaze_trials.csv")
preproc_csv_path = os.path.join(base_dir,"dlc", "2024-06-10_17-53-09_TMaze_preprocessed_positions.csv")
metadata_csv_path= os.path.join(base_dir,"dlc", "2024-06-10_17-53-09_TMaze_metadata.csv")
# --- Load everything ---
trim_info = load_trim_info(trim_info_path)
trials    = load_trials(trials_csv_path)
coords_full, frames_full, pre_df = load_preprocessed_positions(preproc_csv_path,
                                                                x_col="x_smooth",
                                                                y_col="y_smooth")
ports = load_ports_from_metadata(metadata_csv_path)
base, left, right, poly_BL, poly_BR, poly_LR = make_polylines_from_ports(ports)

# --- Build trimmed trajectory ---
loc_trim = build_trimmed_location_from_preprocessed(
    coords_full,
    frames_full,
    trim_start_frame=trim_info["start_frame"],
)

# --- Adjust trials to trimmed frame coordinates ---
trials_adj = adjust_trials_for_trim(
    trials,
    trim_start_frame=trim_info["start_frame"],
    n_frames_trimmed=loc_trim.shape[0],
)

# --- Linearize + bin ---
lin_df, bin_edges = linearize_and_bin_trials(
    location_trim=loc_trim,
    trials_adj=trials_adj,
    base=base,
    left=left,
    right=right,
    poly_BL=poly_BL,
    poly_BR=poly_BR,
    poly_LR=poly_LR,
    n_bins=30,
    save=True,
    out_dir=base_dir,
    out_basename="TMaze_linearized_binned_trials",
)
